# 12 — LE DÉCOMPOSEUR : un audio → sa décomposition automatique 🎯

L'outil bout-en-bout qui assemble toutes les briques du projet. On lui donne **un enregistrement de djembé**, il fait tout seul :
1. **Sépare** les voix (dunun / jembé) par NMF informée,
2. **détecte et type** les frappes du jembé (bass/tone/slap),
3. **identifie le rythme** (Manjanin / Maraka / Woloso),
4. **estime le tempo** et trace la **courbe d'intensité**,
5. sort une **transcription** (événements datés).

> Honnêteté : le typage des frappes se fait sur la voix jembé *séparée* (moins propre que les pistes isolées) → il est indicatif. Séparer soliste et accompagnement dans un mono reste hors de portée.

## 0. Mise en place des modèles (~2–3 min, une fois)

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, librosa
import matplotlib.pyplot as plt
from IPython.display import Audio, display
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import djembe_utils as dj
c = dj.list_corpus()

# dictionnaires spectraux (séparation)
Wd = dj.learn_spectral_dict(c[c['code'].isin(['D1','D2'])].to_dict('records'), dj.KD)
Wj = dj.learn_spectral_dict(c[c['code'].isin(['J1','J2'])].to_dict('records'), dj.KJ)
W = np.hstack([Wd, Wj])

# classifieur de frappes (bass/tone/slap)
st = pd.read_csv('output/all_jembe_strokes_typed.csv')
clf_str = make_pipeline(StandardScaler(), RandomForestClassifier(200, random_state=0)).fit(st[dj.FAST_FEATS], st['stroke'])

def dunun_onsets(y):
    H = dj.nmf_activations(dj.spec(y), W)
    return librosa.onset.onset_detect(onset_envelope=H[:dj.KD].sum(0).astype(np.float32), sr=dj.SR, hop_length=dj.HOP, units='time', delta=0.15, wait=4)
def jembe_onsets(y):
    H = dj.nmf_activations(dj.spec(y), W)
    return librosa.onset.onset_detect(onset_envelope=H[dj.KD:].sum(0).astype(np.float32), sr=dj.SR, hop_length=dj.HOP, units='time', delta=0.15, wait=4)
def ioi_features(on):
    i = np.diff(on); i = i[(i>0.05)&(i<2.5)]
    if len(i) < 20: return None
    m = pd.Series(i).rolling(9, center=True, min_periods=3).median().values; v = i/m
    h, _ = np.histogram(v, bins=np.linspace(0.3,3,14), density=True)
    return list(h) + [np.std(v), np.mean(np.abs(np.log(v[1:]/v[:-1])))]

# classifieur de rythme (IOI du dunun séparé), même pipeline audio pour tout le corpus
rows = []
for (p, t), _ in c.groupby(['piece','take']):
    mix, _ = dj.build_mix(p, int(t)); f = ioi_features(dunun_onsets(mix))
    if f: rows.append({'piece':p, **{f'f{i}':x for i,x in enumerate(f)}})
D = pd.DataFrame(rows); FF = [k for k in D if k.startswith('f')]
clf_ryt = RandomForestClassifier(400, random_state=0).fit(D[FF], D['piece'])
print('Modèles prêts (séparation + frappes + rythme).')

## 1. La fonction `decompose`

In [ ]:
def intensity(y, onsets, win=4.0):
    dur = len(y)/dj.SR; tg = np.arange(0, max(dur-win, win), 1.0)
    rms = librosa.feature.rms(y=y, hop_length=512)[0]
    tr = librosa.frames_to_time(np.arange(len(rms)), sr=dj.SR, hop_length=512)
    dens = np.array([np.sum((onsets>=t)&(onsets<t+win))/win for t in tg])
    r = np.array([rms[(tr>=t)&(tr<t+win)].mean() for t in tg])
    z = lambda x:(x-x.min())/(np.ptp(x)+1e-9)
    return tg, (z(dens)+z(r))/2*100

def decompose(y):
    don, jon = dunun_onsets(y), jembe_onsets(y)
    jf = dj.fast_stroke_features(y, jon)
    strokes = clf_str.predict(jf[dj.FAST_FEATS]) if len(jf) else np.array([])
    fr = ioi_features(don)
    ryt = clf_ryt.predict([fr])[0]; proba = dict(zip(clf_ryt.classes_, clf_ryt.predict_proba([fr])[0]))
    iois = np.diff(don); med = np.median(iois[(iois>0.1)&(iois<1.5)])
    tg, inten = intensity(y, np.sort(np.concatenate([don, jon])))
    trans = pd.concat([pd.DataFrame({'time':don, 'instrument':'dunun', 'stroke':''}),
                       pd.DataFrame({'time':jf['time'].values if len(jf) else [], 'instrument':'jembe', 'stroke':strokes})]).sort_values('time')
    return dict(don=don, jon=jon, strokes=strokes, rythme=ryt, proba=proba, ioi_med=med, tg=tg, inten=inten, trans=trans)

## 2. Test sur un mix du corpus (vérité connue)
Change `PIECE`/`TAKE`. On reconstruit le mix mono et on le décompose comme s'il était inconnu.

In [ ]:
PIECE, TAKE = 'Maraka', 2
mix, codes = dj.build_mix(PIECE, TAKE)
R = decompose(mix)
sc = pd.Series(R['strokes']).value_counts(normalize=True).round(2).to_dict()
print(f'== Décomposition (vérité : {PIECE}, instruments {codes}) ==')
print(f"  dunun : {len(R['don'])} frappes | jembé : {len(R['jon'])} frappes")
print(f"  frappes jembé (approx.) : {sc}")
print(f"  RYTHME : {R['rythme']}  ({ {k:round(v,2) for k,v in R['proba'].items()} })")
print(f"  pulsation dunun ~ {60/R['ioi_med']:.0f} bpm")

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
w = (R['don']>=20)&(R['don']<28); ax[0].vlines(R['don'][w], 0, 1, color='#2b6cff', label='dunun')
wj = (R['jon']>=20)&(R['jon']<28); ax[0].vlines(R['jon'][wj], 1, 2, color='#d63bd6', label='jembé')
ax[0].set(title='Transcription (20–28 s)', xlabel='temps (s)', yticks=[]); ax[0].legend()
ax[1].bar(list(R['proba']), list(R['proba'].values()), color=['#4a8cff','#e05ce0','#3ddb7a']); ax[1].set(title='Rythme (proba)', ylim=(0,1))
ax[2].plot(R['tg'], R['inten'], color='#e05ce0', lw=2); ax[2].fill_between(R['tg'], R['inten'], alpha=.2, color='#e05ce0')
ax[2].set(title='Intensité musicale', xlabel='temps (s)', ylim=(0,105))
plt.tight_layout(); plt.show()
print('\nTranscription (extrait) :'); display(R['trans'][(R['trans']['time']>=20)&(R['trans']['time']<23)].round(3))

## 3. Sur une vidéo YouTube (audio réel, jamais vu)

In [ ]:
y, _ = librosa.load('archive/youtube/Woloso.wav', sr=dj.SR, mono=True)
R = decompose(y)
print('== YouTube Woloso ==')
print(f"  dunun {len(R['don'])} | jembé {len(R['jon'])} frappes")
print(f"  RYTHME prédit : {R['rythme']}  ({ {k:round(v,2) for k,v in R['proba'].items()} })")
print(f"  pulsation ~ {60/R['ioi_med']:.0f} bpm | intensité {R['inten'][:8].mean():.0f} → {R['inten'][-8:].mean():.0f}")
display(Audio(y[int(30*dj.SR):int(42*dj.SR)], rate=dj.SR))

## Bilan — la boucle est bouclée
- `decompose(audio)` fait **tout seul** : séparation → frappes → rythme → tempo → intensité → transcription. Ça marche sur les mix du corpus **et** sur une vidéo YouTube réelle.
- C'est le « modèle qui décompose de lui-même » demandé.
- Limites honnêtes : typage des frappes approximatif sur voix séparée ; pas de séparation soliste/accompagnement en mono ; 3 rythmes connus.
- Il ne reste, pour l'objectif ultime, qu'à **rendre le générateur (07) meilleur** (swing mesuré en 08, structure longue) — le décomposeur, lui, est complet.